# Итоги для кейса VK Case Champ

Синтетические данные за 30 дней сентября 2026 года. Цели кейса — больше бизнесов используют инструменты продаж ВК и больше заказов оформляются внутри ВК. Ниже — опорные выводы из агрегированных таблиц и отдельного анализа `Business_groups.csv`; рекламные расходы рассматриваются только как ограниченный диагностический сигнал.

## Как читать маршруты: методологически корректная трактовка

- **Карточка Маркета VK:** открытие карточки товара Маркета во VK, затем действие пользователя. Нажатие на кнопку может вести к чекауту внутри VK, в чат с продавцом или на другой адрес — фактический справочник назначения кнопок в выгрузке отсутствует.
- **Карточка партнёра во VK:** открытие карточки товара партнёра во VK и последующее нажатие на кнопку. Нажатие может вести к разным следующим шагам, включая оформление на площадке партнёра. Сам факт нажатия не означает автоматически оформленный или оплаченный заказ.
- **Прямой маршрут партнёра:** переход к партнёру без открытия карточки товара во VK. В данных он измеряется отдельным событием и относится к иной когорте.

Партнёрская карточка и прямой переход партнёра нельзя складывать в уникальных покупателей или конверсию. Маркет и партнёра следует сравнивать как связанные, но не одинаковые продуктовые маршруты. Для выводов нужны когортные данные по назначению кнопки, чекауту и оплате.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

data_path = Path.cwd().parent / 'Data'
mv = 'Маркет Вконтакте'
p1 = 'Партнёр 1'
pd.set_option('display.float_format', '{:,.2f}'.format)

market = pd.read_csv(data_path / 'Market_Partner.csv').set_index('Метрика')
traffic = pd.read_csv(data_path / 'Market_Partner_traffic.csv').set_index('Источник')
total = pd.read_csv(data_path / 'Market_Total.csv').set_index('Период')
category = pd.read_csv(data_path / 'Market_Partner_categories.csv').set_index('Категория')
kpi = market.loc[['Показы товаров', 'Открытия карточки товара', 'Нажатия на кнопку в карточке', 'Оформленные заказы', 'GMV, ₽', 'Средний чек, ₽', 'Уникальные покупатели за 30 дней', 'Заказов на покупателя', 'Доля повторных покупателей (2+ заказа), %'], [mv, p1]].copy()
kpi['Доля Маркета в сумме каналов, %'] = np.nan
for metric in ['Показы товаров', 'Открытия карточки товара', 'Нажатия на кнопку в карточке', 'Оформленные заказы', 'GMV, ₽', 'Уникальные покупатели за 30 дней']:
    kpi.loc[metric, 'Доля Маркета в сумме каналов, %'] = kpi.loc[metric, mv] / (kpi.loc[metric, mv] + kpi.loc[metric, p1]) * 100
kpi

,Маркет Вконтакте,Партнёр 1,"Доля Маркета в сумме каналов, %"
Метрика,,,
Показы товаров,"4,777,750,023.00","5,729,628,790.00",45.47
Открытия карточки товара,"193,512,461.00","86,882,822.00",69.01
Нажатия на кнопку в карточке,"5,969,063.00","36,626,075.00",14.01
Оформленные заказы,"413,740.00","4,927,525.00",7.75
"GMV, ₽","1,564,546,740.00","5,119,514,685.00",23.41
"Средний чек, ₽","3,781.00","1,039.00",NaN
Уникальные покупатели за 30 дней,"296,148.00","2,901,334.00",9.26
Заказов на покупателя,1.40,1.70,NaN
"Доля повторных покупателей (2+ заказа), %",19.00,26.60,NaN


In [2]:
# Интерактивные визуализации
import plotly.graph_objects as go

def show_plotly(fig, title, x_title, y_title, height=500):
    fig.update_layout(
        title=dict(text=title), template='plotly_white', height=height,
        xaxis_title=x_title, yaxis_title=y_title,
        legend_title_text='Канал', margin=dict(l=70, r=30, t=80, b=90),
        font=dict(size=13), hoverlabel=dict(namelength=-1),
    )
    fig.show(config={'displaylogo': False, 'responsive': True})


### Интерактивно: вклад Маркета в показатели двух каналов
Доли рассчитаны относительно суммы двух каналов; аудитории могут пересекаться. Это не доля всего рынка.


In [3]:
share_metrics = ['Показы товаров', 'Открытия карточки товара', 'Нажатия на кнопку в карточке',
                 'Оформленные заказы', 'GMV, ₽', 'Уникальные покупатели за 30 дней']
share = kpi.loc[share_metrics, 'Доля Маркета в сумме каналов, %']
fig = go.Figure(go.Bar(x=share.iloc[::-1], y=share.index[::-1], orientation='h', marker_color='#3978C6',
                       hovertemplate='%{y}<br>Доля Маркета: %{x:.2f}%<extra></extra>'))
fig.update_xaxes(range=[0, 100], ticksuffix='%')
show_plotly(fig, 'Масштаб Маркета в сумме двух каналов', 'Доля Маркета, %', '', 500)


## 1. Место ВК среди площадок

Маркет привлекает много открытий, но партнёр значительно превосходит его по числу заказов; у Маркета существенно выше средний чек. Это сравнение только с одним партнёром, а не оценка доли ВК среди всех маркетплейсов и классифайдов. Заказы партнёра могут включать прямые переходы без открытия карточки ВК, а покупки за пределами ВК видны не полностью.
Партнёрский прямой маршрут ведёт на страницу товара партнёра без открытия карточки VK. Нажатие в карточке партнёра может вести к разным следующим шагам, а фактическое назначение кнопок нужно проверить по справочнику событий. Доля нажатий в карточке Маркета не сопоставима с партнёрской как конверсия: следующие шаги различны.


## 2. Для кого и какие боли проверять

Продукты питания и хобби — заметные категории по заказам Маркета. Гипотеза: начать с продавцов, для которых важны сообщество, контент и повторное взаимодействие, а также проверить сценарии локальных продаж. Категория не доказывает ни размер бизнеса, ни его потребности: сегментацию следует проверить интервью; таблица бизнес-сообществ разобрана в 06_business_groups.ipynb.

Отношение нажатий к открытиям и доля учтённых оплат среди оформленных — сигналы для проверки, но не диагноз проблем карточки или чекаута. Кнопки ведут к разным действиям, а внешние оплаты могут не наблюдаться; сначала проверить назначение кнопок и учёт событий.

## 3. УТП и инструменты

Гипотеза УТП: продавать там, где бизнес уже общается с аудиторией, соединяя контент и сообщество с каталогом и простым оформлением заказа. Проверить улучшения карточек, публикаций товаров в ленте и клипах, поиска, чекаута и инструментов повторных продаж. Ключевые метрики экспериментов: активные бизнесы с товарами и продажами, оформленные заказы внутри ВК, а также качество заказов и оплаты. Разделять результат ВК и партнёра; измерять перенос, а не предполагать каннибализацию.

## 4. Роль Маркета

Главная отдельного Маркета даёт небольшую долю открытий, тогда как витрины магазинов и другие поверхности значимы. Предварительно Маркет разумно рассматривать как инфраструктуру каталога, поиска и оформления на разных поверхностях ВК, а не делать единственную ставку на самостоятельную витрину. Это гипотеза о роли продукта, а не доказательство, что отдельная витрина не нужна: «Другое» крупное, данные об исходах по источникам отсутствуют.

In [4]:
sources = traffic.loc[~traffic.index.str.startswith('в т.ч.')]
role = pd.DataFrame({'Открытия Маркета': sources[mv], 'Доля открытий Маркета, %': sources[mv] / sources[mv].sum() * 100, 'Открытия партнёра': sources[p1], 'Доля открытий партнёра, %': sources[p1] / sources[p1].sum() * 100})
role

,Открытия Маркета,"Доля открытий Маркета, %",Открытия партнёра,"Доля открытий партнёра, %"
Источник,,,,
Другое,"77,753,037.00",40.18,5380616,6.19
Витрина магазина,"75,604,323.00",39.07,714540,0.82
Лента,"18,795,696.00",9.71,15622386,17.98
Клипы,"17,239,066.00",8.91,64917109,74.72
Поиск,"3,707,521.00",1.92,159788,0.18
Главная Маркета Вконтакте,"412,818.00",0.21,88383,0.10


### Интерактивно: откуда открывают карточки Маркета
Доли относятся к сумме учтённых источников открытий Маркета. «Другое» сохранено; данных о заказах по источникам нет.


In [5]:
role_plot = role.sort_values('Доля открытий Маркета, %')
fig = go.Figure(go.Bar(x=role_plot['Доля открытий Маркета, %'], y=role_plot.index, orientation='h',
                       marker_color='#3978C6', customdata=role_plot['Открытия Маркета'],
                       hovertemplate='%{y}<br>Доля: %{x:.2f}%<br>Открытий: %{customdata:,.0f}<extra></extra>'))
fig.update_xaxes(rangemode='tozero', ticksuffix='%')
show_plotly(fig, 'Источники открытий карточек Маркета', 'Доля в сумме источников Маркета, %', '', 500)


## 5. Путь бизнеса и проверка гипотез

1. Подключить сообщество и добавить товары: измерить долю бизнесов, завершивших настройку и опубликовавших доступный каталог.
2. Получить качественные показы через сообщество, поиск и контент: проверить видимость и переходы, не складывая пересекающиеся источники.
3. Разделить маршруты Маркета: карточка → чат, чекаут VK или иной переход; исследовать назначения кнопок, условия заказа и оплату на уровне когорт.
4. Вернуть покупателя: тестировать повторные коммуникации и частоту повторных заказов с учётом согласий и качества сервиса.
5. Для партнёра отдельно учитывать путь через карточку VK к чекауту партнёра и прямой переход на страницу товара; проверять статусы и атрибуцию. Экономику и ROMI без сопоставимых затрат и доходов не рассчитывать.

## Ограничения и следующие шаги

- Синтетические данные и один месяц не подтверждают причинность, сезонность или репрезентативность рынка.
- Разные назначения кнопок, прямые переходы партнёра на страницу товара и внешние оплаты нарушают сопоставимость простой воронки. Маршруты команды требуют сверки со справочником событий.
- Суммы заказов по категориям и в разбивках не равны общим итогам; классификацию и покрытие нужно уточнить. «Другое» существенно, а данные по заказам по источникам и GMV по категориям отсутствуют.
- Дневные уникальные пользователи не складываются в месячных уникальных. У партнёра велика доля покупателей без указанного региона.
- Расходы на рекламу в семи агрегированных CSV не анализировались; в Business_groups.csv реклама используется только как описательный признак. Эффективность рекламы и ROMI без атрибуции и затрат по результату оценить нельзя.

## Дополнение: приоритизация бизнес-сообществ

Агрегаты каналов отвечают на вопрос масштаба, но для кейса нужно понять, какие бизнесы готовы к продаже. В таблице Business_groups.csv доступен формат сообщества, активность контента, товары, чекаут, доставка, оплата и реферальная программа. Сегменты ниже — критерии отбора для интервью и пилота, не готовый прогноз роста.

In [6]:
import pandas as pd
groups = pd.read_csv(data_path / 'Business_groups.csv')
for col in ['Подключён чекаут', 'Подключена доставка', 'Подключена оплата',
            'Автор оригинального контента (оценка модели)', 'Участие в реферальной программе']:
    groups[col] = pd.to_numeric(groups[col], errors='coerce').fillna(0)

groups['Готовность к товарной продаже'] = (
    (groups['Товары загружены'] > 0) &
    (groups['Число активных товаров в каталоге'] > 0)
)

theme = groups.groupby('Тематика').agg(
    Бизнесов=('ID бизнеса', 'count'),
    Доля_с_товарным_каталогом=('Готовность к товарной продаже', 'mean'),
    Доля_с_чекаутом=('Подключён чекаут', 'mean'),
    Доля_с_доставкой=('Подключена доставка', 'mean'),
    Доля_с_оплатой=('Подключена оплата', 'mean'),
    Доля_в_реферальной_программе=('Участие в реферальной программе', 'mean'),
).reset_index()
for col in theme.columns:
    if col.startswith('Доля_'):
        theme[col] = theme[col] * 100
theme = theme.nlargest(12, 'Бизнесов')

fig = go.Figure()
for col, label, color in [
    ('Доля_с_товарным_каталогом', 'Товарный каталог', '#3978C6'),
    ('Доля_с_чекаутом', 'Чекаут', '#70A7C6'),
    ('Доля_с_доставкой', 'Доставка', '#E07A5F'),
    ('Доля_с_оплатой', 'Оплата', '#7AA17A'),
]:
    fig.add_trace(go.Bar(
        y=theme['Тематика'], x=theme[col], name=label, orientation='h',
        hovertemplate='%{y}<br>' + label + ': %{x:.1f}%<extra></extra>'
    ))
fig.update_layout(template='plotly_white', barmode='group', height=620,
                  title='Готовность тематик к товарной продаже',
                  xaxis_title='Доля бизнес-сообществ в тематике, %',
                  xaxis=dict(ticksuffix='%', rangemode='tozero'),
                  yaxis=dict(autorange='reversed'),
                  legend=dict(orientation='h', y=1.02),
                  margin=dict(l=180, r=30, t=100, b=60))
fig.show(config={'displaylogo': False, 'responsive': True})

theme.sort_values('Бизнесов', ascending=False)[[
    'Тематика', 'Бизнесов', 'Доля_с_товарным_каталогом',
    'Доля_с_чекаутом', 'Доля_с_доставкой', 'Доля_с_оплатой'
]].style.format({
    'Доля_с_товарным_каталогом': '{:.1f}%',
    'Доля_с_чекаутом': '{:.1f}%',
    'Доля_с_доставкой': '{:.1f}%',
    'Доля_с_оплатой': '{:.1f}%'
})

,Тематика,Бизнесов,Доля_с_товарным_каталогом,Доля_с_чекаутом,Доля_с_доставкой,Доля_с_оплатой
9,Красота и здоровье,1703,24.1%,48.4%,4.6%,1.3%
10,Кулинария и еда,837,56.2%,66.8%,14.3%,2.5%
4,Дом и ремонт,744,49.5%,62.5%,11.8%,3.4%
33,Творчество и хендмейд,538,54.5%,67.1%,17.8%,4.3%
20,Образование,537,38.0%,51.0%,6.0%,2.8%
36,Транспорт,521,39.5%,56.8%,9.2%,2.1%
16,Мода и стиль,460,55.9%,70.7%,15.9%,3.9%
1,Бизнес и работа,436,36.5%,55.0%,8.5%,2.3%
26,Путешествия и отдых,416,46.4%,60.8%,3.4%,2.2%
35,Товары и магазины,405,64.9%,80.0%,20.5%,5.4%


### Итоговая формулировка для презентации

**Роль Маркета:** не заменить партнёра по объёму внешних продаж, а объединить контент, поиск, карточку товара и оформление в один сценарий внутри ВК. Для бизнесов приоритетны те темы и форматы, где уже есть товарный каталог и доступность инструментов продажи.

**Что подтверждать дальше:** назначения конкретных кнопок в карточке Маркета и партнёра, число пользователей, дошедших до оформления и оплаты, долю повторных покупок в когортах и готовность бизнес-сообществ по тематикам.

**Не утверждать без дополнительных данных:** что низкая доля оплат объясняет плохой чекаут; что нажатие в карточке всегда приводит к чекауту; что одна категория однозначно наиболее прибыльная; что прямой переход партнёра и карточка партнёра относятся к одной и той же аудитории.

## Связь с анализом бизнес-сообществ

Подробный анализ `Business_groups.csv` вынесен в [06_business_groups.ipynb](06_business_groups.ipynb). Он показывает сторону предложения: из 9 754 сообществ только 298 имеют хотя бы один заказ через чекаут, поэтому цель кейса — не просто повысить конверсию существующего трафика, а провести бизнесы по разным стадиям готовности к продаже.

Приоритетная модель:

1. **Продавцы через чекаут** — тестовая база для масштабирования контента, реферальных механик и повторных заказов.
2. **Товарные сообщества без чекаута** — основной резерв роста числа бизнесов с продажами внутри ВК.
3. **Крупные сообщества без товарного контура** — сначала каталог, упаковка предложения или сценарий заявок; чекаут не должен быть обязательным первым шагом.

Это дополняет анализ трафика: клипы и реферальные публикации могут приводить внимание, но для бизнеса нужен измеримый переход от контента к карточке, действию, оформлению и оплате. Прямого идентификатора бизнеса между таблицами нет, поэтому связь с категориями, источниками и покупателями — стратегическая, а не причинная.

## Практическая рамка решения

Роль Маркета ВКонтакте — инфраструктура социальной коммерции, а не только отдельная витрина: помочь бизнесу выбрать подходящий путь (каталог, товарный контент, карточка, чат, чекаут или заявка) и измерять результат по этапам. Для сервисных сообществ товарный чекаут не является универсальным KPI.

Рекомендуемые KPI по сегментам: активные сообщества с каталогом; товарные показы; открытия карточек; нажатия с разметкой назначения; переходы к оформлению; оформленные и оплаченные заказы; GMV; повторные покупки; стоимость привлечения. Нажатия нельзя приравнивать к чекауту без отдельного события назначения и оформления.